# 03 · Evaluation results

This notebook reads the files `src/evaluate.py` writes into `results/` and prints the tables the report uses. It runs nothing and makes no LLM call.

- `retrieval.csv`: the retrievers and cross-encoders compared without an LLM (`evaluate.py retrieval`).
- `threshold_sweep.csv`: the similarity threshold scored on the original 30 questions (`evaluate.py sweep`).
- `summary.csv` and `records.jsonl`: the five systems answering the 40 questions (`evaluate.py run --judge`, run day by day by `scripts/run_evaluation.py`).

Question groups: A01–A20 answerable, O01–O10 out of the corpus (used to choose the gate), N01–N10 held out (near-domain, written before any run, used to choose nothing).

In [1]:
import os
import sys

# The notebooks live in notebooks/; the project code is in src/.
ROOT = os.path.abspath("..")
sys.path.insert(0, os.path.join(ROOT, "src"))

import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
RESULTS = os.path.join(ROOT, "results")

## 1. Retrieval and gate comparison (no LLM call)

In [2]:
retrieval = pd.read_csv(os.path.join(RESULTS, "retrieval.csv"))
print(retrieval.to_string(index=False))

         retriever  hit_rate  page_hit_rate  context_precision  auroc  margin  gap_share  heldout_auroc  heldout_margin  mean_latency_ms
             dense      0.95           0.80              0.857  1.000  0.0357      0.074          0.950         -0.0806              144
              bm25      1.00           0.95              0.932  0.945 -5.8951     -0.157          0.635        -24.8051                9
            hybrid      1.00           0.90              0.911  1.000  0.0357      0.074          0.950         -0.0806              158
     hybrid+medcpt      0.95           0.80              0.889  0.995 -0.0001     -0.000          0.975         -0.0002            13089
hybrid+medcpt_gate      1.00           0.90              0.911  1.000  0.2224      0.222          0.970         -0.0013             2714
        hybrid+bge      1.00           0.95              0.908  1.000  0.0633      0.064          1.000          0.0182            49628
   hybrid+bge_gate      1.00           0.

`page_hit_rate`: share of answerable questions whose evidence page is among the 5 passages. `auroc` and `margin`: how well the row's best score separates answerable from out-of-corpus questions. Rows ending in `_gate` keep the hybrid passages and only score them with the cross-encoder. The `heldout_` columns do the same against the held-out questions.

## 2. Threshold sweep of the cosine gate

In [3]:
sweep = pd.read_csv(os.path.join(RESULTS, "threshold_sweep.csv"))
print(sweep.to_string(index=False))

 threshold  answer_rate  refusal_rate  balanced
      0.30         1.00           0.0     0.500
      0.35         1.00           0.1     0.550
      0.40         1.00           0.2     0.600
      0.45         1.00           0.3     0.650
      0.50         1.00           0.3     0.650
      0.55         1.00           0.7     0.850
      0.60         1.00           1.0     1.000
      0.65         0.95           1.0     0.975
      0.70         0.65           1.0     0.825
      0.75         0.25           1.0     0.625
      0.80         0.10           1.0     0.550


## 3. The five systems

In [4]:
summary = pd.read_csv(os.path.join(RESULTS, "summary.csv"))
print(summary.to_string(index=False))

    system                                  model  n  answer_rate  refusal_rate  heldout_refusal_rate  hit_rate  context_precision  faithfulness  n_judged  mean_latency_s
rag_rerank nvidia/nemotron-3-ultra-550b-a55b:free 39         1.00           1.0                   1.0       1.0              0.906           1.0        19            6.03
rag_hybrid nvidia/nemotron-3-ultra-550b-a55b:free 40         0.95           1.0                   1.0       1.0              0.911           1.0        19            5.18
    no_rag nvidia/nemotron-3-ultra-550b-a55b:free  6         1.00           NaN                   NaN       NaN                NaN           NaN         0            9.31


In [5]:
records = pd.read_json(os.path.join(RESULTS, "records.jsonl"), lines=True)
records["group"] = records["id"].str[0].map({"A": "answerable", "O": "out_of_corpus", "N": "held_out"})
print(len(records), "answers recorded of 240 planned")
print(pd.crosstab([records["system"], records["group"]], records["reason"]).to_string())

84 answers recorded of 240 planned
reason                    answered  below_rerank_threshold  below_threshold  model_refused
system     group                                                                          
no_rag     answerable            6                       0                0              0
rag_hybrid answerable           19                       0                0              0
           held_out              0                       0                7              3
           out_of_corpus         0                       0               10              0
rag_rerank answerable           19                       0                0              0
           held_out              0                       5                0              5
           out_of_corpus         0                      10                0              0


`below_threshold` and `below_rerank_threshold`: refused by the gate, with no LLM call. `model_refused`: the LLM wrote the refusal sentence. `uncited`: the reply cited no passage, or a number it was not given, so it was turned into a refusal. `answered`: a cited answer.

While the evaluation is still running, these tables are partial; rerun the notebook when `scripts/run_evaluation.py` prints "evaluation complete".